In [ ]:
import torch
import torch.nn as nn
from torch.nn import functional as F

#stating the hyperparameters -- COMMENTS are influenced by sakshambaurai

batch_size = 32
block_size = 256
max_iters= 3000
eval_interval = 500

learning_rate = 3e-3
device = 'cuda' if torch.cuda.is_available() else 'cpu'

eval_iters = 200 #When checking how well the model is doing, run 200 evaluation batches and average their losses.
n_embd= 384 #How many numbers are used to represent each token internally.
n_head = 6
n_layer = 6
dropout = 0.2


torch.manual_seed(1337)

with open('/content/input.txt','r',encoding='utf-8') as f:
    text = f.read()


In [ ]:
# Unique characters will be listed here
chars = sorted(list(set(text)))
vocab_size = len(chars)

In [ ]:
# creates a mapping from characters to integers
stoi = { ch:i for i,ch in enumerate(chars) }
itos = { i:ch for i,ch in enumerate(chars) }
encode = lambda s: [stoi[c] for c in s] # encoder: take a string, output a list of integers
decode = lambda l: ''.join([itos[i] for i in l]) # decoder: take a list of integers, output a string

In [ ]:
#ENCODING THE WHOLE DATASET INTO A TENSOR
data = torch.tensor(encode(text), dtype=torch.long)

#train_test_split

n = int(0.9*len(data)) # now we seperate the 90% of the data into training and rest will go into val_data = =(for later validation)

train_data =data[:n] #Take the first 900 items.
val_data = data[n:] #Take everything from item 900 until the end


In [ ]:
# data Loader
def get_batch(split):
    # generate a small batch of data from input of x and y
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    x, y = x.to(device), y.to(device)
    return x, y


In [ ]:

@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

In [ ]:
class Head(nn.Module):
    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))

        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        # input of size (batch, time-step, channels)
        # output of size (batch, time-step, head size)
        B,T,C = x.shape
        k = self.key(x)   # (B,T,hs)
        q = self.query(x) # (B,T,hs)
        # compute attention scores ("affinities")
        wei = q @ k.transpose(-2,-1) * k.shape[-1]**-0.5 # (B, T, hs) @ (B, hs, T) -> (B, T, T)
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf')) # (B, T, T)
        wei = F.softmax(wei, dim=-1) # (B, T, T)
        wei = self.dropout(wei)
        # perform the weighted aggregation of the values
        v = self.value(x) # (B,T,hs)
        out = wei @ v # (B, T, T) @ (B, T, hs) -> (B, T, hs)
        return out

In [ ]:
class MultiHeadAttention(nn.Module):

    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(head_size * num_heads, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        out = self.dropout(self.proj(out))
        return out

In [ ]:
class FeedForward(nn.Module):
    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout), # adding dropouts for early stopouts
        )

    def forward(self, x):
        return self.net(x)

# class Block(nn.Module):
#     """"Transformer block: so cross communication can be computed"""

#     def __init__(self,n_embd,n_head):
#         super().__init__()

#         head_size = n_embd // n_head
#         self.sa = MultiHeadAttention(n_head,head_size)
#         self.ffwd = FeedForward(n_embd)
#         self.ln1 = nn.LayerNorm(n_embd)
#         self.ln2 = nn.LayerNorm(n_embd)

#     def forward(self,x):
#         x = x + self.sa(self.ln1(x))
#         x = x + self.ffwd(self.ln2(x))

#         return x



In [ ]:
class Block(nn.Module):
    def __init__(self, n_embd, n_head):
        # n_embd: embedding dimension, n_head: the number of heads we'd like
        super().__init__()
        head_size = n_embd // n_head
        self.sa = MultiHeadAttention(n_head, head_size)
        self.ffwd = FeedForward(n_embd)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x

In [ ]:
# Main language model
class BigramLanguageModel(nn.Module):

    def __init__(self):
        super().__init__()

        # Token embeddings
        self.token_embedding_table = nn.Embedding(
            vocab_size,
            n_embd
        )

        # Position embeddings
        self.position_embedding_table = nn.Embedding(
            block_size,
            n_embd
        )

        # Transformer blocks
        self.blocks = nn.Sequential(
            *[
                Block(n_embd, n_head=n_head)
                for _ in range(n_layer)
            ]
        )

        # Final normalization
        self.ln_f = nn.LayerNorm(n_embd)

        # Convert embeddings to vocabulary logits
        self.lm_head = nn.Linear(
            n_embd,
            vocab_size
        )

    def forward(self, idx, targets=None):

        B, T = idx.shape

        # Get token embeddings
        token_emb = self.token_embedding_table(idx)

        # Get position embeddings
        pos_emb = self.position_embedding_table(
            torch.arange(
                T,
                device=idx.device
            )
        )

        # Combine token and position embeddings
        x = token_emb + pos_emb

        # Run through Transformer blocks
        x = self.blocks(x)

        # Final normalization
        x = self.ln_f(x)

        # Get vocabulary logits
        logits = self.lm_head(x)

        # Calculate loss during training
        if targets is not None:

            B, T, C = logits.shape

            logits = logits.view(B * T, C)
            targets = targets.view(B * T)

            loss = F.cross_entropy(
                logits,
                targets
            )

        else:
            loss = None

        return logits, loss

    def generate(self, idx, max_new_tokens):

        # Generate one token at a time
        for _ in range(max_new_tokens):

            # Keep only the latest block_size tokens
            idx_cond = idx[:, -block_size:]

            # Get model predictions
            logits, loss = self(idx_cond)

            # Only use the last prediction
            logits = logits[:, -1, :]

            # Convert logits to probabilities
            probs = F.softmax(logits, dim=-1)

            # Sample the next token
            idx_next = torch.multinomial(
                probs,
                num_samples=1
            )

            # Add the new token
            idx = torch.cat(
                (idx, idx_next),
                dim=1
            )

        return idx

In [ ]:
7

model = BigramLanguageModel()
m = model.to(device)
# print the number of parameters in the model
print(sum(p.numel() for p in m.parameters())/1e6, 'M parameters')

# create a PyTorch optimizer
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

for iter in range(max_iters):

    # every once in a while evaluate the loss on train and val sets
    if iter % eval_interval == 0 or iter == max_iters - 1:
        losses = estimate_loss()
        print(f"step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")

    # sample a batch of data
    xb, yb = get_batch('train')

    # evaluate the loss
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

# generate from the model
context = torch.zeros((1, 1), dtype=torch.long, device=device)
print(decode(m.generate(context, max_new_tokens=500)[0].tolist()))
#open('more.txt', 'w').write(decode(m.generate(context, max_new_tokens=10000)[0].tolist()))

10.788929 M parameters
step 0: train loss 4.2843, val loss 4.2817
